## Hydrodynamic model of the Southern North Sea

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#IMPORT LIBRARIES
import gc
import mikeio
import modelskill as ms
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cmcrameri.cm as cmc
import pandas as pd
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))
print("Current working directory:", Path.cwd())
print("Parent directory added to path:", Path.cwd().parent.parent)

from helpers import get_wl_point_obs
from helpers import extract_water_level_measurements, eval_all, eval_all_lagged
from helpers import load_hydro_fields, reshape_and_permute_months

from src.datamanager import DataManager, ROMDataManager
from src.engine import SHREDEngine
from src.train import fit, evaluate, lagged_evaluate, lagged_fit
from src.mSHRED.mSHRED import mSHRED
from src.utils import plot_training_curves, evaluate_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Current working directory: /home/matteo/Documents/Masked-SHRED/Notebooks_final/ORESUND
Parent directory added to path: /home/matteo/Documents/Masked-SHRED


## Data Load

In [3]:
# OPEN FOLDERS
dfs_fldr = "datasets/output/Area.dfsu"
msh_fldr = "datasets/input/mesh_EMOD.mesh"
obs_fldr  = "datasets/observations/"

msh = mikeio.open(msh_fldr)
dfs = mikeio.open(dfs_fldr)

# Read one physical field at a time (float32, invalid elements already dropped).
time, fields, _valid = load_hydro_fields(dfs)
H = fields["Surface elevation"]
U = fields["U velocity"]
V = fields["V velocity"]
del fields, dfs
gc.collect()

# sensor coverage
alllist = get_wl_point_obs()

# For sparse data simulation
#hf = ["Drogden", "Klagshamn", "Dragor", "Barseback"]
# availabilities 97-50-98-50
#alllist = [s for s in alllist if s.name in hf]

# Extract the values and convert to torch tensors

measurements = extract_water_level_measurements(alllist, ds_time=time, obs_fldr=obs_fldr)
print("IN:\nMeasurements shape:", measurements.shape)

# print the shape of the arrays
print("OUT:\nH shape:", H.shape)
print("U shape:", U.shape)
print("V shape:", V.shape)

del msh, time, alllist, _valid
gc.collect()

IN:
Measurements shape: (70127, 12)
OUT:
H shape: (70127, 3320)
U shape: (175295, 3320)
V shape: (175295, 3320)


305

In [4]:
# Assuming 30 days per month, 24 hours per day, and measurements every 30 minutes (2 per hour)
lenmonths = (30 * 24 * 2)
n_total = H.shape[0]
nmonths, ntime_keeped, permuted_indices, (H, U, V, measurements) = reshape_and_permute_months(
    H, U, V, measurements, steps_per_month=lenmonths, seed=42
)
print(f"Time keeprd: {ntime_keeped}/{n_total} total steps ({nmonths} months) -> coverage: {100*ntime_keeped/n_total:.1f}% \n")

print("Reshaped H shape:", H.shape)
print("Reshaped U shape:", U.shape)
print("Reshaped V shape:", V.shape)
print("Reshaped measurements shape:", measurements.shape)
print("\npermutation indices:", permuted_indices)

del n_total
gc.collect()

Time keeprd: 69120/70127 total steps (48 months) -> coverage: 98.6% 

Reshaped H shape: (48, 1440, 3320)
Reshaped U shape: (48, 1440, 3320)
Reshaped V shape: (48, 1440, 3320)
Reshaped measurements shape: (48, 1440, 12)

permutation indices: [27 40 26 43 24 37 12 19  4 25  8  3  6 39 33 13 17 45 15  9 16 29 32 46
  0 31 30  5 11 34  1 44 21  2 36 35 23 41 10 22 18 47 20  7 42 14 28 38]


0

In [5]:
ERRORS = []

# ROMDataManager
LAGS = 24

TRAIN_SIZE = 0.8        # percentage of data used for training 
VAL_SIZE = 0.1          # percentage of data used for validation
TEST_SIZE = 0.1         # percentage of data used for testing

COMPRESS_H = 12         # modes kept for H (surface elevation)
COMPRESS_U = 40         # modes kept for U (U velocity)
COMPRESS_V = 68         # modes kept for V (V velocity)

# Training
OPTIMIZER = "AdamW"     # optimizer to use
LEARNING_RATE = 5e-4    # learning rate for training
NUM_EPOCHS = 200        # maximum number of epochs for training
LR_SCHEDULER = "CosineAnnealingLR"      # learning rate scheduler to use
SCHEDULER_STEP_SIZE = NUM_EPOCHS        # step size for learning rate scheduler (if applicable)

BATCH_SIZE = 32         # batch size for training
PATIENCE = 40           # number of epochs with no improvement on validation loss before stopping training
VERBOSE_TRAIN = True

# Architecture
ENCODER_DEPTH = 3               # number of encoder layers
HIDDEN_SIZE = 32                # hidden size for encoder and decoder layers
N_HEADS = 4                     # number of attention heads (if using a transformer-based architecture)
DIM_FEEDFORWARD = 48            # dimension of feedforward network in transformer (if using a transformer-based architecture)
DECODER = "cnn"                 # type of decoder to use (mpl or cnn)   
DECODER_HIDDEN_SIZES = [64,96]     # hidden sizes for decoder layers 
DROPOUT = 0.1                   # dropout rate for encoder and decoder layers
MASKING_TYPE = "mask_data"      # type of masking to apply to inputs during training (mask_data, mask_data_random, etc.)

## Normal training

In [6]:
import json

# 1. open 1-reconstruction_metrics.json
with open("1-reconstruction_metrics-run12.json") as f:
    metrics = json.load(f)

# 2. best model = lowest NMAE on "Unify all" (test split)
SPLIT = "test"
best = min(metrics, key=lambda m: metrics[m][SPLIT]["Unify all"]["nmae"])
print(f"Best model: {best} (NMAE unify all, {SPLIT}) = {metrics[best][SPLIT]['Unify all']['nmae']}")

# 3. write it to 2-reconstruction_metrics.json as "last_step_{model}"
with open("2-reconstruction_metrics.json", "w") as f:
    json.dump({f"last_step_{best}": metrics[best]}, f, indent=2)

Best model: T3F (NMAE unify all, test) = 0.011926480178241113


## Lagged Training (T3F)

In [7]:
# Data manager
manager = ROMDataManager(lags=LAGS, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE)
# add data to manager
manager.add_data(H, id = "Surface elevation" , compress = COMPRESS_H, measurements = measurements)
manager.add_data(U, id = "U velocity", compress = COMPRESS_U)
manager.add_data(V, id = "V velocity", compress = COMPRESS_V)
# prepare datasets
train_dataset, val_dataset, test_dataset = manager.prepare(verbose = True, pad_output=True, masking_type=MASKING_TYPE)

# Compressed states live in the manager; measurements are no longer needed as a separate copy.
del measurements
gc.collect()

# network input and output shapes
nn_input_shape = train_dataset.X.shape[-1]
nn_output_shape = train_dataset.Y.shape[-1] 
print("\nInput shape for the model:", nn_input_shape)
print("Output shape for the model:", nn_output_shape)


Datasets: ['Surface elevation', 'U velocity', 'V velocity']

Number of nan values in sensor measurements after masking: 233523

Data shapes after preparation:
Train:	 torch.Size([54720, 24, 12]) -> torch.Size([54720, 24, 120])
Val:	 torch.Size([7200, 24, 12]) -> torch.Size([7200, 24, 120])
Test:	 torch.Size([7200, 24, 12]) -> torch.Size([7200, 24, 120])

Input shape for the model: 12
Output shape for the model: 120


In [ ]:
# mSHRED TRAINING
model_args = dict(
    # Encoder
    encoder="transformer_learned_mask", encoder_depth=ENCODER_DEPTH, input_size=nn_input_shape, n_heads=N_HEADS,
    dim_feedforward=DIM_FEEDFORWARD, hidden_size=HIDDEN_SIZE, input_length=LAGS, double_channels = False,
    # Decoder
    decoder=DECODER, decoder_hidden_sizes=DECODER_HIDDEN_SIZES, output_size=nn_output_shape, dropout=DROPOUT, decoder_3D = True
)
mshred = mSHRED(args=model_args).to(device)

W_FACTOR = 0.8
W_KEEPED_TIMESTEPS = 16

# check if the model is already trained and load the weights if available
model_path = Path("models_saved/T3F_LAGGED.pth")
if model_path.exists():
    print(f"Loading pre-trained model from {model_path}")
    mshred.load_state_dict(torch.load(model_path, map_location=device))

else:

    # Use cosine annealing LR schedule
    train_errors, valid_errors = lagged_fit(
        mshred, train_dataset, val_dataset,
        optimizer=OPTIMIZER, lr=LEARNING_RATE, num_epochs=NUM_EPOCHS, patience=PATIENCE, verbose=VERBOSE_TRAIN,
        lr_scheduler=LR_SCHEDULER, scheduler_step_size=SCHEDULER_STEP_SIZE, w_factor=W_FACTOR, w_keeped_steps=W_KEEPED_TIMESTEPS, masking_type=MASKING_TYPE,
    )

    # Save the trained model
    model_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(mshred.state_dict(), model_path)

eval_all_lagged(manager, mshred, train_dataset, val_dataset, test_dataset, H, U, V, model_name="multi_step_T3F")

del model_args, model_path, mshred, H, U, V
gc.collect()

Loading pre-trained model from models_saved/T3F_LAGGED.pth
